# broadcasting-rules — ex4: insert a missing axis where broadcast fails

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `broadcasting-rules`. When a test cell passes, your progress is reported back to your account.

**What you'll practice.** Five broadcasting patterns that ramp from predicting the result shape → row-vector broadcast → column-vector broadcast → targeted axis insertion → outer product via broadcast. Read the docstring, fill the function body, run the test cell. The solution sits in the collapsed `<details>` block below each exercise.

**Per-exercise structure** (Doughty et al. ACE 2024 — `[Bloom level] + [LO] + [Keywords] + [KCs]`):
Each exercise begins with a yaml block stating its Bloom cognitive level, learning objective, keywords, and the knowledge components (KCs) it targets. This makes the cognitive demand explicit instead of buried.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Numpy: Vectorization and broadcasting` subtopic.
You can copy the token from your Delta Drills account page.

This drill exercises the **atom `broadcasting-rules`**, which bridges to the bank subtopic `Numpy: Vectorization and broadcasting` for EWMA state. Completing all 5 exercises triggers a single `arena-rating` beacon at the end of the notebook.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "broadcasting-rules"
DD_SUBTOPIC = "Numpy: Vectorization and broadcasting"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

# Track which exercises passed in this session.
_dd_passed = set()

## Broadcasting — quick refresher

**The rule** (NumPy & PyTorch agree):
1. Right-align both shapes; left-pad the shorter with 1s.
2. For each pair of aligned axes: equal → keep; one is 1 → use the other; otherwise → incompatible.

**Three patterns you reach for constantly:**
- **Row broadcast** — `(N, D) + (D,)` works automatically. Adds a per-feature bias.
- **Column broadcast** — `(N, D) * w` where `w` is `(N,)` fails. Reshape `w` to `(N, 1)` first.
- **Axis insertion** — `unsqueeze` / `[:, None]` / `reshape` are all valid ways to insert a size-1 axis where broadcasting needs it.

### Exercise 4 — insert a missing axis where broadcast fails

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Bloom level: Apply
> LO: Apply targeted axis insertion (`unsqueeze`) to make a 3-D + 1-D broadcast work along the desired axis.
> Keywords: unsqueeze, axis-insertion, shape-debug
> ```

**KCs targeted:** `broadcast-via-unsqueeze`

Implement `ex4_scale_channel(x, w)` to scale each feature channel of a feature-map batch by a per-channel weight.

Input shapes: `x` is `(B, C, H, W)`, `w` is `(C,)`. Output shape: `(B, C, H, W)`. Each `out[b, c, :, :] == x[b, c, :, :] * w[c]`.

Right-align would try to broadcast `(C,)` against the trailing `W` axis — wrong. Reshape `w` to insert size-1 axes where they need to be so the broadcast targets the channel axis instead.

**Hint:** the right shape for `w` to broadcast against `(B, C, H, W)` along the channel axis is `(1, C, 1, 1)`. Use `w.reshape(...)` or chained `unsqueeze` calls — both are fine.

In [ ]:
def ex4_scale_channel(x: Tensor, w: Tensor) -> Tensor:
    return x * w.reshape(1, -1, 1, 1)


<details><summary>Solution</summary>

```python
def ex4_scale_channel(x: Tensor, w: Tensor) -> Tensor:
    return x * w.reshape(1, -1, 1, 1)
```

**Three equivalent forms** for inserting axes:
- `w.reshape(1, -1, 1, 1)` — explicit; `-1` infers C.
- `w[None, :, None, None]` — slice-syntax `None` is shorthand for `unsqueeze`.
- `w.unsqueeze(0).unsqueeze(-1).unsqueeze(-1)` — chained, error-prone in higher dims.

Pick whichever reads clearest at the call site. `reshape` is usually the most explicit for >2 axis insertions.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex4'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex4',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',  # single-exercise standalone — neutral signal
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()